In [1]:
import mlflow
import dagshub

C:\Users\KIIT0001\AppData\Roaming\Python\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
mlflow.set_tracking_uri("https://dagshub.com/Aayush10671/mental-health-score-predictor.mlflow")

In [3]:
import dagshub
dagshub.init(repo_owner='Aayush10671', repo_name='mental-health-score-predictor', mlflow=True)

import mlflow
with mlflow.start_run():
  mlflow.log_param('parameter name', 'value')
  mlflow.log_metric('metric name', 1)

Accessing as Aayush10671

Initialized MLflow to track repo "Aayush10671/mental-health-score-predictor"

Repository Aayush10671/mental-health-score-predictor initialized!

🏃 View run dapper-cat-838 at: https://dagshub.com/Aayush10671/mental-health-score-predictor.mlflow/#/experiments/0/runs/4bc463a2d52340bf9fdcec06064707e0
🧪 View experiment at: https://dagshub.com/Aayush10671/mental-health-score-predictor.mlflow/#/experiments/0


In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder, FunctionTransformer,LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline



In [5]:
mlflow.set_experiment("experiment-3--hyperparameter-tuning")

2026/08/01 22:41:57 INFO mlflow.tracking.fluent: Experiment with name 'experiment-3--hyperparameter-tuning' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/8f0e4282113d4db8bbf8f4c6ba48fc0c', creation_time=1785604317748, experiment_id='3', last_update_time=1785604317748, lifecycle_stage='active', name='experiment-3--hyperparameter-tuning', tags={}, workspace='default'>

In [6]:
df = pd.read_csv('../data/raw/dataset.csv')

In [7]:
num_feature = df.select_dtypes(include = 'number')
len(num_feature.columns)

7

In [8]:
df.shape

(5000, 13)

In [9]:
Q1 = num_feature.quantile(0.25)
Q3 = num_feature.quantile(0.75)
IQR = Q3 - Q1

lower_limit = Q1 - 1.5 * IQR
upper_limit = Q3 + 1.5 * IQR

outliers = ((num_feature < lower_limit) | (num_feature > upper_limit)).sum()
outliers


Age                         0
Avg_Daily_Usage_Hours       0
Daily_Unlocks               0
Study_Hours                 2
Physical_Activity_Hours    22
Sleep_Hours_Per_Night       0
Mental_Health_Score         0
dtype: int64

In [10]:
df.drop_duplicates(inplace=True)

df['Physical_Activity_Hours'] = df['Physical_Activity_Hours'].clip(lower=0)

In [11]:
top_countries = df['Country'].value_counts().head(11)
def group_countries(country):
    if country in top_countries.index:
        return country
    else:
        return 'Other'

In [12]:
df['Grouped_Country'] = df['Country'].apply(group_countries)

In [14]:
df = df[~df.isin(outliers).any(axis=1)]

In [15]:
skewed_col = ['Study_Hours']
other_numric_col = ['Age', 'Avg_Daily_Usage_Hours', 'Sleep_Hours_Per_Night', 'Physical_Activity_Hours' , 'Daily_Unlocks']
ordinal_col = ['Stress_Level']

normal_col = ['Gender','Academic_Level','Most_Used_Platform','Grouped_Country' , 'Purpose_Of_Use']  
features_col = skewed_col + other_numric_col + ordinal_col + normal_col

X = df[features_col]
y = df['Mental_Health_Score']

In [16]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder,LabelEncoder,OrdinalEncoder, FunctionTransformer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

skew_pipeline = Pipeline(steps=[('log_transformer', FunctionTransformer(np.log1p, validate=True)), ('scaler', StandardScaler())])
                                
plain_pipeline = Pipeline(steps=[('scaler', StandardScaler())])

ordinal_pipeline = Pipeline(steps=[('ordinal_encoder', OrdinalEncoder(categories=[['Low', 'Medium', 'High' , 'Very High']]))])

nominal_pipeline = Pipeline(steps=[('onehot_encoder', OneHotEncoder(drop='first' , handle_unknown='ignore'))])


preprocessor = ColumnTransformer(transformers=[
    ('skew', skew_pipeline, skewed_col),
    ('plain', plain_pipeline, other_numric_col),
    ('ordinal', ordinal_pipeline, ordinal_col),
    ('nominal', nominal_pipeline, normal_col)
])

In [17]:
from sklearn.model_selection import GridSearchCV

In [19]:


param_grid = {
    "regressor__n_estimators": [100, 200],
    "regressor__max_depth": [None, 10, 20],
    "regressor__min_samples_split": [2, 5],
    "regressor__min_samples_leaf": [1, 2]
}

with mlflow.start_run():

    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=0.2,
        random_state=42
    )

    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("regressor", RandomForestRegressor(random_state=42))
    ])

    grid_search = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grid,
        cv=5,
        scoring="r2",
        n_jobs=-1
    )

    grid_search.fit(X_train, y_train)

    # Log every parameter combination
    for i in range(len(grid_search.cv_results_["params"])):

        params = grid_search.cv_results_["params"][i]
        mean_cv_score = grid_search.cv_results_["mean_test_score"][i]
        std_cv_score = grid_search.cv_results_["std_test_score"][i]

        model = Pipeline([
            ("preprocessor", preprocessor),
            ("regressor", RandomForestRegressor(
                random_state=42,
                n_estimators=params["regressor__n_estimators"],
                max_depth=params["regressor__max_depth"],
                min_samples_split=params["regressor__min_samples_split"],
                min_samples_leaf=params["regressor__min_samples_leaf"]
            ))
        ])

        model.fit(X_train, y_train)

        y_pred = model.predict(X_test)

        mse = mean_squared_error(y_test, y_pred)
        rmse = np.sqrt(mse)
        r2 = r2_score(y_test, y_pred)

        with mlflow.start_run(nested=True, run_name=f"model_{i}"):

            mlflow.log_params(params)

            mlflow.log_metric("cv_mean_r2", mean_cv_score)
            mlflow.log_metric("cv_std_r2", std_cv_score)

            mlflow.log_metric("test_mse", mse)
            mlflow.log_metric("test_rmse", rmse)
            mlflow.log_metric("test_r2", r2)

            mlflow.sklearn.log_model(model, f"model_{i}")

            print("=" * 60)
            print(f"Model {i+1}")
            print(params)
            print(f"CV Mean R² : {mean_cv_score:.4f}")
            print(f"Test MSE   : {mse:.4f}")
            print(f"Test R²    : {r2:.4f}")

    # Best model
    best_model = grid_search.best_estimator_

    y_pred = best_model.predict(X_test)

    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, y_pred)

    mlflow.log_params(grid_search.best_params_)
    mlflow.log_metric("best_cv_r2", grid_search.best_score_)
    mlflow.log_metric("best_test_mse", mse)
    mlflow.log_metric("best_test_rmse", rmse)
    mlflow.log_metric("best_test_r2", r2)

    mlflow.sklearn.log_model(best_model, "best_model")

    print("\n" + "=" * 60)
    print("🏆 BEST MODEL")
    print("=" * 60)
    print("Best Parameters:", grid_search.best_params_)
    print(f"Best CV R² : {grid_search.best_score_:.4f}")
    print(f"Test MSE   : {mse:.4f}")
    print(f"Test RMSE  : {rmse:.4f}")
    print(f"Test R²    : {r2:.4f}")

2026/08/01 22:50:01 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/08/01 22:50:06 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Model 1
{'regressor__max_depth': None, 'regressor__min_samples_leaf': 1, 'regressor__min_samples_split': 2, 'regressor__n_estimators': 100}
CV Mean R² : 0.8605
Test MSE   : 0.1968
Test R²    : 0.8897
🏃 View run model_0 at: https://dagshub.com/Aayush10671/mental-health-score-predictor.mlflow/#/experiments/3/runs/d4c2fc6148c842ca9833807f487f0618
🧪 View experiment at: https://dagshub.com/Aayush10671/mental-health-score-predictor.mlflow/#/experiments/3


2026/08/01 22:53:02 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/08/01 22:53:06 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Model 2
{'regressor__max_depth': None, 'regressor__min_samples_leaf': 1, 'regressor__min_samples_split': 2, 'regressor__n_estimators': 200}
CV Mean R² : 0.8613
Test MSE   : 0.1940
Test R²    : 0.8913
🏃 View run model_1 at: https://dagshub.com/Aayush10671/mental-health-score-predictor.mlflow/#/experiments/3/runs/a2603fad224b435a90ec2ad5387d62d9
🧪 View experiment at: https://dagshub.com/Aayush10671/mental-health-score-predictor.mlflow/#/experiments/3


2026/08/01 22:58:02 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/08/01 22:58:07 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


...

Model 23
{'regressor__max_depth': 20, 'regressor__min_samples_leaf': 2, 'regressor__min_samples_split': 5, 'regressor__n_estimators': 100}
CV Mean R² : 0.8510
Test MSE   : 0.2164
Test R²    : 0.8788
🏃 View run model_22 at: https://dagshub.com/Aayush10671/mental-health-score-predictor.mlflow/#/experiments/3/runs/2319bc41f0a148118c04f4c52466188a
🧪 View experiment at: https://dagshub.com/Aayush10671/mental-health-score-predictor.mlflow/#/experiments/3


2026/08/01 23:28:39 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/08/01 23:28:44 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Model 24
{'regressor__max_depth': 20, 'regressor__min_samples_leaf': 2, 'regressor__min_samples_split': 5, 'regressor__n_estimators': 200}
CV Mean R² : 0.8520
Test MSE   : 0.2140
Test R²    : 0.8801
🏃 View run model_23 at: https://dagshub.com/Aayush10671/mental-health-score-predictor.mlflow/#/experiments/3/runs/d2a9740c1ca3490bb86a0749154d4d1c
🧪 View experiment at: https://dagshub.com/Aayush10671/mental-health-score-predictor.mlflow/#/experiments/3


2026/08/01 23:29:59 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/08/01 23:30:04 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html



🏆 BEST MODEL
Best Parameters: {'regressor__max_depth': None, 'regressor__min_samples_leaf': 1, 'regressor__min_samples_split': 2, 'regressor__n_estimators': 200}
Best CV R² : 0.8613
Test MSE   : 0.1940
Test RMSE  : 0.4405
Test R²    : 0.8913
🏃 View run upset-doe-728 at: https://dagshub.com/Aayush10671/mental-health-score-predictor.mlflow/#/experiments/3/runs/90deb9cd474e4e9dadbe458dcf93ea6f
🧪 View experiment at: https://dagshub.com/Aayush10671/mental-health-score-predictor.mlflow/#/experiments/3
